In [3]:
import pandas as pd

## Collected csv

In [22]:
import pandas as pd


def analyze_trading_csv(csv_path, max_loss_sl=5):
    """
    Analyze trading CSV using an R-based daily loss limit.

    max_loss_sl:
        Maximum allowed daily loss expressed in number of SLs.
        Example:
            5 = stop trading at -5R

    Returns:
        analytics: dictionary containing all analysis results
        filtered_df: dataframe with trades after daily shutdown removed
    """

    df = pd.read_csv(csv_path)

    # -----------------------------
    # Basic cleanup
    # -----------------------------
    df["Date"] = pd.to_datetime(
        df["Date"],
        dayfirst=True,
        errors="coerce"
    ).dt.date

    df["Result"] = (
        df["Result"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Preserve original order
    df["_original_order"] = range(len(df))

    # -----------------------------
    # Calculate R for every trade
    # -----------------------------
    def calculate_r(row):

        try:
            entry = float(row["Entry"])
            sl = float(row["SL"])
            exit_price = float(row["Exit"])

            risk = abs(entry - sl)

            if risk == 0:
                return 0.0

            if str(row["Side"]).upper() == "BUY":
                return (exit_price - entry) / risk

            elif str(row["Side"]).upper() == "SELL":
                return (entry - exit_price) / risk

            return 0.0

        except (ValueError, TypeError):
            return 0.0

    df["R"] = df.apply(calculate_r, axis=1)

    # -----------------------------
    # Original statistics
    # -----------------------------
    total_sl = (df["Result"] == "SL").sum()
    total_tp = (df["Result"] == "TP").sum()
    total_be = (df["Result"] == "BE").sum()

    total_trades = len(df)

    # -----------------------------
    # Find daily shutdown points
    # -----------------------------
    shutdown_dates = []
    shutdown_details = []

    ignored_indices = set()

    for date, day_df in df.groupby("Date", sort=True):

        cumulative_r = 0.0
        shutdown_index = None

        for index, row in day_df.iterrows():

            cumulative_r += row["R"]

            if cumulative_r <= -max_loss_sl:
                shutdown_index = index

                shutdown_dates.append(date)

                shutdown_details.append({
                    "Date": date,
                    "ShutdownTradeNumber":
                        list(day_df.index).index(index) + 1,
                    "CumulativeR": cumulative_r,
                    "SL": (day_df.loc[:index, "Result"] == "SL").sum(),
                    "TP": (day_df.loc[:index, "Result"] == "TP").sum(),
                    "BE": (day_df.loc[:index, "Result"] == "BE").sum(),
                    "TradesIgnoredAfterShutdown":
                        len(day_df.loc[index:]) - 1
                })

                # Ignore all trades AFTER shutdown trade
                for later_index in day_df.index:
                    if later_index > index:
                        ignored_indices.add(later_index)

                break

    # -----------------------------
    # Filter later trades
    # -----------------------------
    filtered_df = df.drop(index=list(ignored_indices)).copy()

    # -----------------------------
    # Filtered statistics
    # -----------------------------
    filtered_sl = (filtered_df["Result"] == "SL").sum()
    filtered_tp = (filtered_df["Result"] == "TP").sum()
    filtered_be = (filtered_df["Result"] == "BE").sum()

    filtered_total = len(filtered_df)

    filtered_r = filtered_df["R"].sum()

    # -----------------------------
    # Analytics
    # -----------------------------
    analytics = {

        "Original": {
            "Total Trades": total_trades,
            "SL": int(total_sl),
            "TP": int(total_tp),
            "BE": int(total_be),
        },

        "Daily Loss Limit": {
            "Max Loss (SLs)": max_loss_sl,
            "Max Loss (R)": -max_loss_sl,
        },

        "Shutdown Dates": shutdown_dates,

        "Shutdown Details": pd.DataFrame(
            shutdown_details
        ),

        "Filtered": {
            "Total Trades": filtered_total,
            "SL": int(filtered_sl),
            "TP": int(filtered_tp),
            "BE": int(filtered_be),
            "Total R": round(filtered_r, 2),
        },

        "Ignored Trades": len(ignored_indices),
    }

    return analytics, filtered_df

In [25]:
analytics, filtered_df = analyze_trading_csv(
    r"D:\Fin\Prod\Live Trading\collected data\XAGUSD\SILVER_M15_IST_20260517_20260824.csv",
    max_loss_sl=5
)

In [26]:
print("Original:")
print(analytics["Original"])

print("\nShutdown Dates:")
print(analytics["Shutdown Dates"])

print("\nShutdown Details:")
print(analytics["Shutdown Details"])

print("\nFiltered:")
print(analytics["Filtered"])

print("\nIgnored Trades:")
print(analytics["Ignored Trades"])

Original:
{'Total Trades': 180, 'SL': 84, 'TP': 96, 'BE': 0}

Shutdown Dates:
[]

Shutdown Details:
Empty DataFrame
Columns: []
Index: []

Filtered:
{'Total Trades': 180, 'SL': 84, 'TP': 96, 'BE': 0, 'Total R': np.float64(108.0)}

Ignored Trades:
0


## Backtested

In [27]:
import pandas as pd


def analyze_trading_csv_backtested(csv_path, max_loss_sl=5):
    """
    Analyze a trading CSV and identify days where the daily loss limit
    was reached.

    max_loss_sl:
        Maximum allowed loss expressed as number of stop-losses.

        Example:
            max_loss_sl=5
            means stop trading when cumulative realized R <= -5R.

    Returns:
        analytics      -> dictionary containing the analysis
        filtered_df    -> dataframe with trades after shutdown removed
    """

    # ============================================================
    # LOAD CSV
    # ============================================================

    df = pd.read_csv(csv_path)

    # Preserve original trade order
    df["_original_order"] = range(len(df))

    # ============================================================
    # CLEAN COLUMNS
    # ============================================================

    df["Signal date"] = pd.to_datetime(
        df["Signal date"],
        dayfirst=True,
        errors="coerce"
    ).dt.date

    df["TP/SL outcome"] = (
        df["TP/SL outcome"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["Side"] = (
        df["Side"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Numeric columns
    numeric_columns = [
        "Entry price",
        "Exit price",
        "SL",
        "TP",
        "BE",
        "Gross profit/loss",
        "Commission",
        "Clearing fee",
        "Exchange fee",
        "IP fee",
        "NFA fee",
        "Net profit/loss",
        "MAE",
        "MFE",
        "ETD",
        "Bars held",
        "Duration minutes",
        "Max RR",
        "Cumulative net profit",
    ]

    for column in numeric_columns:

        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # ============================================================
    # CALCULATE R
    # ============================================================

    def calculate_r(row):

        try:

            entry = float(row["Entry price"])
            sl = float(row["SL"])
            exit_price = float(row["Exit price"])

            risk = abs(entry - sl)

            if risk == 0:
                return 0.0

            if row["Side"] == "BUY":
                return (exit_price - entry) / risk

            elif row["Side"] == "SELL":
                return (entry - exit_price) / risk

            return 0.0

        except (ValueError, TypeError):

            return 0.0

    df["R"] = df.apply(calculate_r, axis=1)

    # ============================================================
    # ORIGINAL STATISTICS
    # ============================================================

    total_trades = len(df)

    total_sl = (
        df["TP/SL outcome"] == "SL"
    ).sum()

    total_tp = (
        df["TP/SL outcome"] == "TP"
    ).sum()

    total_be = (
        df["TP/SL outcome"] == "BE"
    ).sum()

    # ============================================================
    # FIND DAILY SHUTDOWNS
    # ============================================================

    shutdown_dates = []
    shutdown_details = []

    ignored_indices = set()

    for date, day_df in df.groupby(
        "Signal date",
        sort=True
    ):

        cumulative_r = 0.0
        shutdown_index = None

        # Make sure trades are evaluated in trade order
        day_df = day_df.sort_values(
            "_original_order"
        )

        for trade_number, (index, row) in enumerate(
            day_df.iterrows(),
            start=1
        ):

            cumulative_r += row["R"]

            # Daily loss limit reached
            if cumulative_r <= -max_loss_sl:

                shutdown_index = index

                shutdown_dates.append(date)

                trades_before_shutdown = (
                    day_df.loc[:index]
                )

                trades_after_shutdown = (
                    len(day_df)
                    - len(trades_before_shutdown)
                )

                shutdown_details.append({

                    "Date": date,

                    "Shutdown Trade":
                        trade_number,

                    "Trade Number":
                        row["Trade number"],

                    "Cumulative R":
                        round(cumulative_r, 2),

                    "SL Count":
                        (
                            trades_before_shutdown[
                                "TP/SL outcome"
                            ] == "SL"
                        ).sum(),

                    "TP Count":
                        (
                            trades_before_shutdown[
                                "TP/SL outcome"
                            ] == "TP"
                        ).sum(),

                    "BE Count":
                        (
                            trades_before_shutdown[
                                "TP/SL outcome"
                            ] == "BE"
                        ).sum(),

                    "Net P/L At Shutdown":
                        round(
                            trades_before_shutdown[
                                "Net profit/loss"
                            ].sum(),
                            2
                        ),

                    "Trades Ignored":
                        trades_after_shutdown,
                })

                # Ignore ONLY trades after the shutdown trade
                shutdown_position = (
                    day_df.index.get_loc(index)
                )

                later_indices = day_df.index[
                    shutdown_position + 1:
                ]

                ignored_indices.update(
                    later_indices
                )

                break

    # ============================================================
    # FILTER DATA
    # ============================================================

    filtered_df = df.drop(
        index=list(ignored_indices)
    ).copy()

    # ============================================================
    # FILTERED STATISTICS
    # ============================================================

    filtered_total = len(filtered_df)

    filtered_sl = (
        filtered_df["TP/SL outcome"] == "SL"
    ).sum()

    filtered_tp = (
        filtered_df["TP/SL outcome"] == "TP"
    ).sum()

    filtered_be = (
        filtered_df["TP/SL outcome"] == "BE"
    ).sum()

    filtered_net_profit = (
        filtered_df["Net profit/loss"].sum()
    )

    filtered_r = filtered_df["R"].sum()

    # ============================================================
    # ANALYTICS
    # ============================================================

    analytics = {

        "Original": {

            "Total Trades":
                int(total_trades),

            "Total SL":
                int(total_sl),

            "Total TP":
                int(total_tp),

            "Total BE":
                int(total_be),

            "Total Net P/L":
                round(
                    df["Net profit/loss"].sum(),
                    2
                ),
        },

        "Daily Loss Rule": {

            "Maximum SL Equivalent":
                max_loss_sl,

            "Maximum Allowed R":
                -max_loss_sl,
        },

        "Shutdown": {

            "Number Of Shutdown Days":
                len(shutdown_dates),

            "Dates":
                shutdown_dates,

            "Details":
                pd.DataFrame(
                    shutdown_details
                ),
        },

        "After Removing Later Trades": {

            "Total Trades":
                int(filtered_total),

            "Total SL":
                int(filtered_sl),

            "Total TP":
                int(filtered_tp),

            "Total BE":
                int(filtered_be),

            "Total Net P/L":
                round(
                    filtered_net_profit,
                    2
                ),

            "Total R":
                round(
                    filtered_r,
                    2
                ),
        },

        "Removed": {

            "Trades Removed":
                len(ignored_indices),
        }
    }

    return analytics, filtered_df

In [28]:
analytics, filtered_df = analyze_trading_csv_backtested(
    r"D:\Fin\Prod\Live Trading\backtesting\Output\output.csv",
    max_loss_sl=5
)

In [29]:
print("ORIGINAL")
print(analytics["Original"])

print("\nDAILY LOSS RULE")
print(analytics["Daily Loss Rule"])

print("\nSHUTDOWN")
print("Number of shutdown days:",
      analytics["Shutdown"]["Number Of Shutdown Days"])

print("Dates:")
for date in analytics["Shutdown"]["Dates"]:
    print(date)

print("\nSHUTDOWN DETAILS")
print(analytics["Shutdown"]["Details"])

print("\nAFTER REMOVING LATER TRADES")
print(analytics["After Removing Later Trades"])

print("\nREMOVED")
print(analytics["Removed"])

ORIGINAL
{'Total Trades': 180, 'Total SL': 76, 'Total TP': 104, 'Total BE': 0, 'Total Net P/L': np.float64(10.56)}

DAILY LOSS RULE
{'Maximum SL Equivalent': 5, 'Maximum Allowed R': -5}

SHUTDOWN
Number of shutdown days: 0
Dates:

SHUTDOWN DETAILS
Empty DataFrame
Columns: []
Index: []

AFTER REMOVING LATER TRADES
{'Total Trades': 180, 'Total SL': 76, 'Total TP': 104, 'Total BE': 0, 'Total Net P/L': np.float64(10.56), 'Total R': np.float64(132.0)}

REMOVED
{'Trades Removed': 0}


# Backtesting configuration to ignore overlapping trades.

In [ ]:
from __future__ import annotations

import argparse
import csv
import math
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal


Signal = Literal["BUY", "SELL"]
Result = Literal["TP", "SL", "BE", "OPEN"]


# ============================================================
# CONFIGURATION
# ============================================================

@dataclass(frozen=True)
class BacktestConfig:
    sl_distance: float = 0.06
    tp_distance: float = 0.24

    be_enabled: bool = False
    be_trigger: float = 0.4
    be_offset: float = 0.0

    position_size: float = 1.0

    output_path: Path | None = None


# ============================================================
# DATA CLASSES
# ============================================================

@dataclass(frozen=True)
class Bar:
    timestamp: int
    open: float
    high: float
    low: float
    close: float
    volume: float


@dataclass(frozen=True)
class SignalRow:
    row_number: int
    date: str
    time: str
    side: str
    timestamp: int | None
    error: str = ""


@dataclass
class BacktestReport:
    output_path: Path
    rows: list[dict[str, object]]
    warnings: list[str]


# ============================================================
# HELPERS
# ============================================================

def _timestamp(value: str) -> int:
    value = value.strip()

    formats = (
        "%d-%m-%Y %H:%M:%S",
        "%d-%m-%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in formats:
        try:
            return int(
                datetime.strptime(
                    value,
                    date_format
                ).replace(
                    tzinfo=timezone.utc
                ).timestamp()
            )
        except ValueError:
            continue

    raise ValueError(
        f"invalid timestamp '{value}'"
    )


def _number(value: str, field: str) -> float:
    try:
        parsed = float(
            value.strip()
        )
    except (AttributeError, ValueError):
        raise ValueError(
            f"invalid {field} '{value}'"
        ) from None

    if not math.isfinite(parsed):
        raise ValueError(
            f"invalid {field} '{value}'"
        )

    return parsed


def _headers(
    row: dict[str, str]
) -> dict[str, str]:

    return {
        key.strip().lower():
            (value or "").strip()
        for key, value in row.items()
    }


def _format_timestamp(
    value: int | None
        ) -> str:

    if value is None:
        return ""

    return datetime.fromtimestamp(
        value,
        timezone.utc
    ).strftime(
        "%d-%m-%Y %H:%M"
    )



# ============================================================
# LOAD FINE TIMEFRAME DATA
# ============================================================

def _load_bars(
    path: Path
    ) -> list[Bar]:

    with path.open(
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as source:

        reader = csv.DictReader(source)

        bars: list[Bar] = []

        for row_number, raw_row in enumerate(
            reader,
            2
        ):

            row = _headers(raw_row)

            raw_time = (
                row.get("time")
                or row.get("datetime")
                or row.get("date")
            )

            if not raw_time:
                raise ValueError(
                    f"FineTimeframe row {row_number}: "
                    "missing time/datetime/date"
                )

            try:

                bars.append(
                    Bar(
                        _timestamp(raw_time),

                        _number(
                            row.get("open", ""),
                            "open"
                        ),

                        _number(
                            row.get("high", ""),
                            "high"
                        ),

                        _number(
                            row.get("low", ""),
                            "low"
                        ),

                        _number(
                            row.get("close", ""),
                            "close"
                        ),

                        _number(
                            row.get("volume", "0")
                            or "0",
                            "volume"
                        ),
                    )
                )

            except ValueError as error:

                raise ValueError(
                    f"FineTimeframe row {row_number}: "
                    f"{error}"
                ) from None

    return sorted(
        bars,
        key=lambda bar: bar.timestamp
    )


# ============================================================
# LOAD SIGNALS
# ============================================================

def _load_signals(
    path: Path
) -> list[SignalRow]:

    with path.open(
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as source:

        reader = csv.DictReader(source)

        signals: list[SignalRow] = []

        for row_number, raw_row in enumerate(
            reader,
            2
        ):

            row = _headers(raw_row)

            date = row.get(
                "date",
                ""
            )

            time = row.get(
                "time",
                ""
            )

            side = row.get(
                "side",
                ""
            ).upper()

            error = ""

            timestamp: int | None = None

            try:

                if not date or not time:
                    raise ValueError(
                        "missing Date or Time"
                    )

                timestamp = _timestamp(
                    f"{date} {time}"
                )

                if side not in (
                    "BUY",
                    "SELL"
                ):
                    raise ValueError(
                        "Side must be BUY or SELL"
                    )

            except ValueError as caught:

                error = str(caught)

            signals.append(
                SignalRow(
                    row_number,
                    date,
                    time,
                    side,
                    timestamp,
                    error
                )
            )

    return signals


# ============================================================
# PRICE MOVEMENT
# ============================================================

def _move(
    side: Signal,
    entry: float,
    bar: Bar,
    favorable: bool
) -> float:

    value = (
        bar.high - entry
        if side == "BUY"
        else entry - bar.low
    )

    if not favorable:

        value = (
            entry - bar.low
            if side == "BUY"
            else bar.high - entry
        )

    return max(
        0.0,
        value
    )


# ============================================================
# SIMULATE ONE TRADE
# ============================================================

def _simulate(
    signal: SignalRow,
    bars: list[Bar],
    config: BacktestConfig
) -> dict[str, object]:

    assert signal.timestamp is not None

    # --------------------------------------------------------
    # Find exact signal candle.
    # --------------------------------------------------------

    entry_index = next(
        (
            index
            for index, bar in enumerate(bars)
            if bar.timestamp == signal.timestamp
        ),
        None
    )

    if entry_index is None:

        raise ValueError(
            "no FineTimeframe candle exactly "
            "matches the signal timestamp"
        )

    # --------------------------------------------------------
    # Validate SL / TP.
    # --------------------------------------------------------

    if config.sl_distance <= 0:

        raise ValueError(
            "SL distance must be greater than zero"
        )

    if config.tp_distance <= 0:

        raise ValueError(
            "TP distance must be greater than zero"
        )

    side: Signal = signal.side  # type: ignore[assignment]

    entry_bar = bars[entry_index]

    entry = entry_bar.open

    # --------------------------------------------------------
    # Calculate SL / TP.
    # --------------------------------------------------------

    sl = (
        entry - config.sl_distance
        if side == "BUY"
        else entry + config.sl_distance
    )

    tp = (
        entry + config.tp_distance
        if side == "BUY"
        else entry - config.tp_distance
    )

    be = (
        entry + config.be_offset
        if side == "BUY"
        else entry - config.be_offset
    )

    max_favorable = 0.0
    max_adverse = 0.0

    be_activated = False

    exit_bar: Bar | None = None

    exit_price = entry

    result: Result = "OPEN"

    bars_held = 0

    # ========================================================
    # TRADE SIMULATION
    # ========================================================

    for bar in bars[entry_index:]:

        bars_held += 1

        # ----------------------------------------------------
        # MFE / MAE
        # ----------------------------------------------------

        max_favorable = max(
            max_favorable,
            _move(
                side,
                entry,
                bar,
                True
            )
        )

        max_adverse = max(
            max_adverse,
            _move(
                side,
                entry,
                bar,
                False
            )
        )

        # ----------------------------------------------------
        # Current stop.
        # ----------------------------------------------------

        stop = (
            be
            if be_activated
            else sl
        )

        # ----------------------------------------------------
        # Check SL.
        # ----------------------------------------------------

        stop_touched = (
            bar.low <= stop
            if side == "BUY"
            else bar.high >= stop
        )

        # ----------------------------------------------------
        # Check TP.
        # ----------------------------------------------------

        target_touched = (
            bar.high >= tp
            if side == "BUY"
            else bar.low <= tp
        )

        # ----------------------------------------------------
        # Exit.
        #
        # If both SL and TP are touched in the same M1 candle,
        # preserve the original behavior:
        #
        # SL wins.
        # ----------------------------------------------------

        if stop_touched or target_touched:

            if (
                stop_touched
                and target_touched
            ):

                event = "SL"

            else:

                event = (
                    "TP"
                    if target_touched
                    else "SL"
                )

            exit_bar = bar

            result = (
                "TP"
                if event == "TP"
                else (
                    "BE"
                    if be_activated
                    else "SL"
                )
            )

            exit_price = (
                tp
                if event == "TP"
                else stop
            )

            break

        # ----------------------------------------------------
        # Break-even activation.
        # ----------------------------------------------------

        if (
            config.be_enabled
            and not be_activated
            and _move(
                side,
                entry,
                bar,
                True
            ) >= config.be_trigger
        ):

            be_activated = True

    # ========================================================
    # TRADE RESULT
    # ========================================================

    exit_timestamp = (
        exit_bar.timestamp
        if exit_bar is not None
        else None
    )

    duration = (
        round(
            (
                exit_timestamp
                - signal.timestamp
            ) / 60
        )
        if exit_timestamp is not None
        else 0
    )

    signed_move = (
        exit_price - entry
        if side == "BUY"
        else entry - exit_price
    )

    gross = (
        signed_move
        * config.position_size
        if exit_bar is not None
        else 0.0
    )

    return {

        "Entry time":
            _format_timestamp(
                signal.timestamp
            ),

        "Entry price":
            entry,

        "Exit time":
            (
                _format_timestamp(
                    exit_timestamp
                )
                if exit_timestamp is not None
                else ""
            ),

        "Exit price":
            exit_price,

        "Unit":
            "USD",

        "SL":
            sl,

        "TP":
            tp,

        "BE":
            (
                be
                if config.be_enabled
                else ""
            ),

        "Exit reason":
            result,

        "TP/SL outcome":
            result,

        "Gross profit/loss":
            gross,

        "Commission":
            0.0,

        "Clearing fee":
            0.0,

        "Exchange fee":
            0.0,

        "IP fee":
            0.0,

        "NFA fee":
            0.0,

        "Net profit/loss":
            gross,

        "MAE":
            max_adverse,

        "MFE":
            max_favorable,

        "ETD":
            "",

        "Bars held":
            bars_held,

        "Duration minutes":
            duration,

        "Max RR":
            round(
                max_favorable
                / config.sl_distance,
                2
            ),

        "BE activated":
            be_activated,

        "Status":
            "OK",

        "Error":
            "",

        # ----------------------------------------------------
        # INTERNAL VALUE
        #
        # Used only by run_backtest().
        # Not written to output CSV.
        # ----------------------------------------------------

        "_exit_timestamp":
            exit_timestamp,
    }


# ============================================================
# OUTPUT COLUMNS
# ============================================================

OUTPUT_FIELDS = [

    "Trade number",

    "Signal date",

    "Signal time",

    "Side",

    "Entry time",

    "Entry price",

    "Exit time",

    "Exit price",

    "SL",

    "TP",

    "BE",

    "Unit",

    "Exit reason",

    "TP/SL outcome",

    "Gross profit/loss",

    "Commission",

    "Clearing fee",

    "Exchange fee",

    "IP fee",

    "NFA fee",

    "Net profit/loss",

    "MAE",

    "MFE",

    "ETD",

    "Bars held",

    "Duration minutes",

    "Max RR",

    "BE activated",

    "Cumulative net profit",

    "Status",

    "Error",
]


# ============================================================
# MAIN BACKTEST
# ============================================================

def run_backtest(
    signal_csv: str | Path,
    finetimeframe_csv: str | Path,
    config: BacktestConfig | None = None
) -> BacktestReport:

    config = (
        config
        if config is not None
        else BacktestConfig()
    )

    signal_path = Path(
        signal_csv
    )

    fine_path = Path(
        finetimeframe_csv
    )

    # ========================================================
    # LOAD DATA
    # ========================================================

    bars = _load_bars(
        fine_path
    )

    signals = _load_signals(
        signal_path
    )

    # ========================================================
    # SORT SIGNALS CHRONOLOGICALLY
    #
    # This is important because the active-trade rule must
    # always process signals in time order.
    #
    # Invalid signals are retained but valid signals are
    # processed chronologically.
    # ========================================================

    signals = sorted(
        signals,
        key=lambda signal: (
            signal.timestamp
            if signal.timestamp is not None
            else float("inf"),
            signal.row_number
        )
    )

    # ========================================================
    # OUTPUT PATH
    # ========================================================

    output_path = (
        config.output_path
        if config.output_path is not None
        else signal_path.with_name(
            f"{signal_path.stem}_trade_outcomes.csv"
        )
    )

    rows: list[
        dict[str, object]
    ] = []

    warnings: list[str] = []

    cumulative = 0.0

    # ========================================================
    # ACTIVE TRADE STATE
    # ========================================================
    #
    # This is the ONLY logic needed for your requirement.
    #
    # None:
    #     No trade is currently active.
    #
    # Timestamp:
    #     A trade is active and will remain active until this
    #     timestamp.
    #
    # Example:
    #
    # Trade #1 exits at 05:50.
    #
    # Any signal:
    #
    # 05:45 -> ignored
    # 05:46 -> ignored
    # 05:49 -> ignored
    #
    # 05:50 -> allowed
    # 05:51 -> allowed
    #
    # ========================================================

    active_trade_exit_timestamp: int | None = None

    active_trade_number: int | None = None

    # ========================================================
    # PROCESS SIGNALS
    # ========================================================

    for trade_number, signal in enumerate(
        signals,
        1
    ):

        row: dict[str, object] = {

            "Trade number":
                trade_number,

            "Signal date":
                signal.date,

            "Signal time":
                signal.time,

            "Side":
                signal.side,
        }

        # ====================================================
        # INVALID SIGNAL
        # ====================================================

        if signal.error:

            row.update({

                "Status":
                    "INVALID_SIGNAL",

                "Error":
                    signal.error,

            })

            warnings.append(
                f"Signal row "
                f"{signal.row_number}: "
                f"{signal.error}"
            )

            row[
                "Cumulative net profit"
            ] = cumulative

            rows.append(
                row
            )

            continue

        assert signal.timestamp is not None

        # ====================================================
        # ACTIVE TRADE CHECK
        # ====================================================
        #
        # If there is an active trade and the new signal occurs
        # BEFORE its exit timestamp, ignore the new signal.
        #
        # At exactly the exit timestamp the old trade is already
        # considered closed, so the new signal is allowed.
        # ====================================================

        if (
            active_trade_exit_timestamp
            is not None
            and signal.timestamp
            < active_trade_exit_timestamp
        ):

            row.update({

                "Entry time":
                    "",

                "Entry price":
                    "",

                "Exit time":
                    "",

                "Exit price":
                    "",

                "SL":
                    "",

                "TP":
                    "",

                "BE":
                    "",

                "Unit":
                    "USD",

                "Exit reason":
                    "IGNORED",

                "TP/SL outcome":
                    "IGNORED",

                "Gross profit/loss":
                    0.0,

                "Commission":
                    0.0,

                "Clearing fee":
                    0.0,

                "Exchange fee":
                    0.0,

                "IP fee":
                    0.0,

                "NFA fee":
                    0.0,

                "Net profit/loss":
                    0.0,

                "MAE":
                    "",

                "MFE":
                    "",

                "ETD":
                    "",

                "Bars held":
                    0,

                "Duration minutes":
                    0,

                "Max RR":
                    "",

                "BE activated":
                    False,

                "Status":
                    "IGNORED",

                "Error":
                    (
                        "Previous trade "
                        f"#{active_trade_number} "
                        "is still active. "
                        f"It exits at "
                        f"{_format_timestamp(active_trade_exit_timestamp)}. "
                        "New signal occurs at "
                        f"{_format_timestamp(signal.timestamp)}."
                    ),
            })

            # ------------------------------------------------
            # IMPORTANT:
            #
            # Ignored trade does NOT:
            #
            # - change active trade
            # - change exit timestamp
            # - change cumulative P/L
            # ------------------------------------------------

            row[
                "Cumulative net profit"
            ] = cumulative

            rows.append(
                row
            )

            continue

        # ====================================================
        # NO ACTIVE TRADE
        #
        # OR
        #
        # Previous trade has already closed.
        #
        # Therefore this signal can be accepted.
        # ====================================================

        try:

            result = _simulate(
                signal,
                bars,
                config
            )

            # ------------------------------------------------
            # Extract internal exit timestamp.
            # ------------------------------------------------

            current_exit_timestamp = result.get(
                "_exit_timestamp"
            )

            # ------------------------------------------------
            # Remove internal field.
            # ------------------------------------------------

            result.pop(
                "_exit_timestamp",
                None
            )

            row.update(
                result
            )

            # ------------------------------------------------
            # Update P/L.
            # ------------------------------------------------

            cumulative += float(
                row.get(
                    "Net profit/loss",
                    0.0
                )
                or 0.0
            )

            # =================================================
            # UPDATE ACTIVE TRADE
            # =================================================
            #
            # If trade has an SL/TP/BE exit:
            #
            #     active_trade_exit_timestamp = exit time
            #
            # If trade somehow remains OPEN:
            #
            #     active_trade_exit_timestamp = None
            #
            # =================================================

            if (
                current_exit_timestamp
                is not None
            ):

                active_trade_exit_timestamp = int(
                    current_exit_timestamp
                )

                active_trade_number = (
                    trade_number
                )

            else:

                active_trade_exit_timestamp = None

                active_trade_number = None

        except ValueError as error:

            row.update({

                "Status":
                    "MISSING_MARKET_DATA",

                "Error":
                    str(error),

                "Net profit/loss":
                    0.0,
            })

            warnings.append(
                f"Signal row "
                f"{signal.row_number}: "
                f"{error}"
            )

        # ====================================================
        # CUMULATIVE P/L
        # ====================================================

        row[
            "Cumulative net profit"
        ] = cumulative

        rows.append(
            row
        )

    # ========================================================
    # WRITE CSV
    # ========================================================

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as destination:

        writer = csv.DictWriter(
            destination,
            fieldnames=OUTPUT_FIELDS,
            extrasaction="ignore"
        )

        writer.writeheader()

        writer.writerows(
            rows
        )

    return BacktestReport(
        output_path,
        rows,
        warnings
    )


# ============================================================
# BACKWARD-COMPATIBLE ALIAS
# ============================================================

Backtest = run_backtest


# ============================================================
# BACKTEST INPUTS
# ============================================================

SIGNAL_CSV = Path(
    r"D:\Fin\Prod\Live Trading\signals recorded\SILVER_M15_IST_20260517_20260824.csv"
)

FINETIMEFRAME_CSV = Path(
    r"D:\Fin\Prod\Live Trading\public\data\SILVER_M1_IST_20260517_20260824.csv"
)

OUTPUT_CSV = Path(
    r"D:\Fin\Prod\Live Trading\backtesting\Output\output.csv"
)


# ============================================================
# MAIN
# ============================================================

def main():

    report = run_backtest(
        SIGNAL_CSV,
        FINETIMEFRAME_CSV,

        BacktestConfig(
            output_path=OUTPUT_CSV,

            sl_distance=0.06,
            tp_distance=0.12,

            be_enabled=False,
            be_trigger=0.4,
            be_offset=0.0,

            position_size=1.0,
        )
    )

    print()
    print("=" * 70)
    print("BACKTEST COMPLETE")
    print("=" * 70)

    print(
        f"Output file: "
        f"{report.output_path}"
    )

    accepted = sum(
        1
        for row in report.rows
        if row.get("Status") == "OK"
    )

    ignored = sum(
        1
        for row in report.rows
        if row.get("Status") == "IGNORED"
    )

    invalid = sum(
        1
        for row in report.rows
        if row.get("Status") == "INVALID_SIGNAL"
    )

    missing_data = sum(
        1
        for row in report.rows
        if row.get("Status") == "MISSING_MARKET_DATA"
    )

    final_cumulative = 0.0

    if report.rows:

        final_cumulative = float(
            report.rows[-1].get(
                "Cumulative net profit",
                0.0
            )
            or 0.0
        )

    print(
        f"Accepted trades      : "
        f"{accepted}"
    )

    print(
        f"Ignored trades       : "
        f"{ignored}"
    )

    print(
        f"Invalid signals      : "
        f"{invalid}"
    )

    print(
        f"Missing market data  : "
        f"{missing_data}"
    )

    print(
        f"Final net P/L        : "
        f"{final_cumulative}"
    )

    print("=" * 70)

    if report.warnings:

        print()
        print("WARNINGS:")

        for warning in report.warnings:

            print(
                f"WARNING: {warning}"
            )


if __name__ == "__main__":
    main()


BACKTEST COMPLETE
Output file: D:\Fin\Prod\Live Trading\backtesting\Output\output.csv
Accepted trades      : 170
Ignored trades       : 1
Invalid signals      : 0
Missing market data  : 0
Final net P/L        : 5.999999999999822


In [32]:
def main():

    report = run_backtest(
        SIGNAL_CSV,
        FINETIMEFRAME_CSV,

        BacktestConfig(
            output_path=OUTPUT_CSV,
            ignore_if_previous_exit_in_current_bracket=True,
        ),
    )

    print()
    print("=" * 70)
    print("BACKTEST COMPLETE")
    print("=" * 70)

    print(
        f"Wrote {len(report.rows)} rows to:"
    )

    print(
        report.output_path
    )

    accepted = sum(
        1
        for row in report.rows
        if row.get("Status") == "OK"
    )

    ignored = sum(
        1
        for row in report.rows
        if row.get("Status") == "IGNORED"
    )

    invalid = sum(
        1
        for row in report.rows
        if row.get("Status") == "INVALID_SIGNAL"
    )

    missing_data = sum(
        1
        for row in report.rows
        if row.get("Status") == "MISSING_MARKET_DATA"
    )

    final_cumulative = 0.0

    if report.rows:
        final_cumulative = float(
            report.rows[-1].get(
                "Cumulative net profit",
                0.0
            ) or 0.0
        )

    print()
    print(f"Accepted trades       : {accepted}")
    print(f"Ignored trades        : {ignored}")
    print(f"Invalid signals       : {invalid}")
    print(f"Missing market data   : {missing_data}")
    print(f"Final cumulative P/L  : {final_cumulative}")

    print("=" * 70)

    for warning in report.warnings:
        print(
            f"WARNING: {warning}"
        )


if __name__ == "__main__":
    main()

TypeError: BacktestConfig.__init__() got an unexpected keyword argument 'ignore_if_previous_exit_in_current_bracket'

In [1]:
import os

folder_names = [
    "XAUUSD",
    "XAGUSD",
    "BTCUSD",
    "ETHUSD",
    "EURUSD",
    "GBPUSD",
    "USDCHF",
    "NASDAQ",
    "S&P 500",
    "RUSSELL",
    "CRUDE"
]

directory = input("Enter the directory path: ").strip()

if not os.path.isdir(directory):
    print(f"Directory does not exist: {directory}")
    exit()

for folder_name in folder_names:
    folder_path = os.path.join(directory, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_path}")

print("\nDone! All folders have been created.")

Created: D:\Fin\Prod\Live Trading\collected data\XAUUSD
Created: D:\Fin\Prod\Live Trading\collected data\XAGUSD
Created: D:\Fin\Prod\Live Trading\collected data\BTCUSD
Created: D:\Fin\Prod\Live Trading\collected data\ETHUSD
Created: D:\Fin\Prod\Live Trading\collected data\EURUSD
Created: D:\Fin\Prod\Live Trading\collected data\GBPUSD
Created: D:\Fin\Prod\Live Trading\collected data\USDCHF
Created: D:\Fin\Prod\Live Trading\collected data\NASDAQ
Created: D:\Fin\Prod\Live Trading\collected data\S&P 500
Created: D:\Fin\Prod\Live Trading\collected data\RUSSELL
Created: D:\Fin\Prod\Live Trading\collected data\CRUDE

Done! All folders have been created.


## MFE discovery

In [3]:
from __future__ import annotations

import argparse
import csv
import math
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal


# ============================================================
# CONFIGURATION
# ============================================================

@dataclass(frozen=True)
class MFEConfig:
    """
    Configuration for the TP-follow-through MFE calculation.
    """

    # Original trade distances.
    sl_distance: float = 0.06
    tp_distance: float = 0.12

    # If True, the MFE calculation may use the candle that
    # actually hits TP.
    #
    # Recommended: True.
    #
    # The TP-hit candle's high/low is part of the price movement
    # that occurred before the eventual return to entry/SL.
    include_tp_hit_candle: bool = True

    # Output file.
    output_path: Path | None = None


# ============================================================
# DATA CLASSES
# ============================================================

Signal = Literal["BUY", "SELL"]


@dataclass(frozen=True)
class Bar:
    timestamp: int
    open: float
    high: float
    low: float
    close: float
    volume: float


@dataclass
class MFEReport:
    output_path: Path
    rows: list[dict[str, object]]
    warnings: list[str]


# ============================================================
# HELPERS
# ============================================================

def _timestamp(value: str) -> int:
    """
    Convert supported date/time formats to UTC timestamp.
    """

    value = value.strip()

    formats = (
        "%d-%m-%Y %H:%M:%S",
        "%d-%m-%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in formats:

        try:

            return int(
                datetime.strptime(
                    value,
                    date_format
                ).replace(
                    tzinfo=timezone.utc
                ).timestamp()
            )

        except ValueError:
            continue

    raise ValueError(
        f"invalid timestamp '{value}'"
    )


def _number(
    value: str,
    field: str
) -> float:

    try:

        parsed = float(
            value.strip()
        )

    except (AttributeError, ValueError):

        raise ValueError(
            f"invalid {field} '{value}'"
        ) from None

    if not math.isfinite(parsed):

        raise ValueError(
            f"invalid {field} '{value}'"
        )

    return parsed


def _headers(
    row: dict[str, str]
) -> dict[str, str]:

    return {
        key.strip().lower():
            (value or "").strip()
        for key, value in row.items()
    }


def _format_timestamp(
    value: int | None
) -> str:

    if value is None:
        return ""

    return datetime.fromtimestamp(
        value,
        timezone.utc
    ).strftime(
        "%d-%m-%Y %H:%M"
    )


# ============================================================
# LOAD M1 DATA
# ============================================================

def _load_bars(
    path: Path
) -> list[Bar]:

    with path.open(
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as source:

        reader = csv.DictReader(source)

        bars: list[Bar] = []

        for row_number, raw_row in enumerate(
            reader,
            2
        ):

            row = _headers(raw_row)

            raw_time = (
                row.get("time")
                or row.get("datetime")
                or row.get("date")
            )

            if not raw_time:

                raise ValueError(
                    f"M1 row {row_number}: "
                    "missing time/datetime/date"
                )

            try:

                bars.append(
                    Bar(
                        timestamp=_timestamp(
                            raw_time
                        ),

                        open=_number(
                            row.get("open", ""),
                            "open"
                        ),

                        high=_number(
                            row.get("high", ""),
                            "high"
                        ),

                        low=_number(
                            row.get("low", ""),
                            "low"
                        ),

                        close=_number(
                            row.get("close", ""),
                            "close"
                        ),

                        volume=_number(
                            row.get("volume", "0")
                            or "0",
                            "volume"
                        ),
                    )
                )

            except ValueError as error:

                raise ValueError(
                    f"M1 row {row_number}: "
                    f"{error}"
                ) from None

    return sorted(
        bars,
        key=lambda bar: bar.timestamp
    )


# ============================================================
# LOAD EXISTING TRADE DATA
# ============================================================

def _load_trades(
    path: Path
) -> list[dict[str, str]]:

    with path.open(
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as source:

        reader = csv.DictReader(source)

        if reader.fieldnames is None:

            raise ValueError(
                "Trade CSV has no header."
            )

        rows: list[dict[str, str]] = []

        for raw_row in reader:

            row = {
                key.strip():
                    (value or "").strip()
                for key, value in raw_row.items()
                if key is not None
            }

            rows.append(row)

    return rows


# ============================================================
# FIND NEXT SIGNAL
# ============================================================

def _get_signal_timestamp(
    row: dict[str, str]
) -> int | None:

    date = (
        row.get("Signal date", "")
        or row.get("signal date", "")
    ).strip()

    time = (
        row.get("Signal time", "")
        or row.get("signal time", "")
    ).strip()

    if not date or not time:

        return None

    try:

        return _timestamp(
            f"{date} {time}"
        )

    except ValueError:

        return None


# ============================================================
# FAVORABLE MOVE
# ============================================================

def _favorable_move(
    side: Signal,
    entry: float,
    bar: Bar
) -> float:

    if side == "BUY":

        return max(
            0.0,
            bar.high - entry
        )

    return max(
        0.0,
        entry - bar.low
    )


# ============================================================
# ADVERSE VISIT
# ============================================================

def _visited_entry_or_sl(
    side: Signal,
    entry: float,
    sl: float,
    bar: Bar
) -> bool:

    if side == "BUY":

        # Price has returned to entry or gone through SL.
        return bar.low <= entry

    # SELL:
    #
    # Price has returned to entry or gone through SL.
    return bar.high >= entry


# ============================================================
# FIND TP HIT
# ============================================================

def _find_tp_hit(
    bars: list[Bar],
    start_index: int,
    end_timestamp: int | None,
    side: Signal,
    tp: float
) -> int | None:
    """
    Find the first M1 candle that actually touches TP.

    Search is restricted to the current trade's signal window.

    The search stops before the next signal.

    If TP and the next signal occur on the same timestamp,
    the current trade is allowed to use that candle.
    """

    for index in range(
        start_index,
        len(bars)
    ):

        bar = bars[index]

        # ----------------------------------------------------
        # Do not cross into the next signal.
        # ----------------------------------------------------

        if (
            end_timestamp is not None
            and bar.timestamp > end_timestamp
        ):

            break

        if side == "BUY":

            if bar.high >= tp:

                return index

        else:

            if bar.low <= tp:

                return index

    return None


# ============================================================
# CALCULATE NEW MFE
# ============================================================

def _calculate_tp_mfe(
    bars: list[Bar],
    entry_index: int,
    next_signal_timestamp: int | None,
    side: Signal,
    entry: float,
    sl: float,
    tp: float,
    config: MFEConfig
) -> tuple[float | None, int | None, int | None, str]:
    """
    Calculate the requested TP-based MFE.

    RULE:

        TP must first be hit.

        Once TP is hit:

            continue tracking favorable movement

        until:

            price returns to entry
            OR
            price reaches/passes SL
            OR
            next signal begins.

    MFE is the maximum favorable excursion measured from
    the original entry price.
    """

    # ========================================================
    # STEP 1
    # FIND TP HIT
    # ========================================================

    tp_hit_index = _find_tp_hit(
        bars=bars,
        start_index=entry_index,
        end_timestamp=next_signal_timestamp,
        side=side,
        tp=tp
    )

    if tp_hit_index is None:

        return (
            None,
            None,
            None,
            "TP_NOT_FOUND"
        )

    # ========================================================
    # STEP 2
    # TRACK MAXIMUM FAVORABLE MOVEMENT
    # ========================================================

    max_favorable = 0.0

    stop_index: int | None = None

    stop_reason = "NEXT_SIGNAL"

    first_index = tp_hit_index

    if not config.include_tp_hit_candle:

        first_index += 1

    for index in range(
        first_index,
        len(bars)
    ):

        bar = bars[index]

        # ----------------------------------------------------
        # NEVER CROSS NEXT SIGNAL.
        # ----------------------------------------------------

        if (
            next_signal_timestamp is not None
            and bar.timestamp >= next_signal_timestamp
        ):

            stop_index = index
            stop_reason = "NEXT_SIGNAL"

            break

        # ----------------------------------------------------
        # MFE
        #
        # We measure favorable movement from the original
        # entry price.
        # ----------------------------------------------------

        favorable = _favorable_move(
            side,
            entry,
            bar
        )

        max_favorable = max(
            max_favorable,
            favorable
        )

        # ----------------------------------------------------
        # RETURN TO ENTRY / SL
        # ----------------------------------------------------
        #
        # Important:
        #
        # We check this AFTER capturing the current candle's
        # favorable excursion.
        #
        # Therefore if a candle first reaches a new high and
        # later trades back to entry within the same candle,
        # that candle's high is included in MFE.
        # ----------------------------------------------------

        if _visited_entry_or_sl(
            side,
            entry,
            sl,
            bar
        ):

            stop_index = index
            stop_reason = "ENTRY_OR_SL"

            break

    return (
        max_favorable,
        tp_hit_index,
        stop_index,
        stop_reason
    )


# ============================================================
# MAIN MFE PROCESSOR
# ============================================================

def recalculate_mfe(
    trades_csv: str | Path,
    finetimeframe_csv: str | Path,
    config: MFEConfig | None = None
) -> MFEReport:

    config = (
        config
        if config is not None
        else MFEConfig()
    )

    trades_path = Path(
        trades_csv
    )

    fine_path = Path(
        finetimeframe_csv
    )

    # ========================================================
    # LOAD
    # ========================================================

    trades = _load_trades(
        trades_path
    )

    bars = _load_bars(
        fine_path
    )

    if not bars:

        raise ValueError(
            "M1 data contains no candles."
        )

    warnings: list[str] = []

    output_rows: list[
        dict[str, object]
    ] = []

    # ========================================================
    # PROCESS EVERY TRADE
    # ========================================================

    for row_index, original_row in enumerate(
        trades
    ):

        row: dict[str, object] = dict(
            original_row
        )

        outcome = (
            row.get("TP/SL outcome", "")
            or row.get("Exit reason", "")
            or ""
        ).strip().upper()

        # ====================================================
        # ONLY TP TRADES
        # ====================================================

        if outcome != "TP":

            output_rows.append(
                row
            )

            continue

        # ====================================================
        # SIDE
        # ====================================================

        side_text = (
            row.get("Side", "")
            or ""
        ).strip().upper()

        if side_text not in (
            "BUY",
            "SELL"
        ):

            warnings.append(
                f"Trade row {row_index + 2}: "
                f"invalid side '{side_text}'"
            )

            output_rows.append(
                row
            )

            continue

        side: Signal = side_text  # type: ignore[assignment]

        # ====================================================
        # ENTRY TIME
        # ====================================================

        entry_time_text = (
            row.get("Entry time", "")
            or ""
        ).strip()

        if not entry_time_text:

            warnings.append(
                f"Trade row {row_index + 2}: "
                "missing Entry time"
            )

            output_rows.append(
                row
            )

            continue

        try:

            entry_timestamp = _timestamp(
                entry_time_text
            )

        except ValueError as error:

            warnings.append(
                f"Trade row {row_index + 2}: "
                f"{error}"
            )

            output_rows.append(
                row
            )

            continue

        # ====================================================
        # ENTRY PRICE
        # ====================================================

        try:

            entry = _number(
                str(
                    row.get(
                        "Entry price",
                        ""
                    )
                ),
                "Entry price"
            )

        except ValueError as error:

            warnings.append(
                f"Trade row {row_index + 2}: "
                f"{error}"
            )

            output_rows.append(
                row
            )

            continue

        # ====================================================
        # SL / TP
        # ====================================================
        #
        # We calculate these independently rather than relying
        # on the CSV values.
        # ====================================================

        if side == "BUY":

            sl = (
                entry
                - config.sl_distance
            )

            tp = (
                entry
                + config.tp_distance
            )

        else:

            sl = (
                entry
                + config.sl_distance
            )

            tp = (
                entry
                - config.tp_distance
            )

        # ====================================================
        # FIND ENTRY CANDLE
        # ====================================================

        entry_index = next(
            (
                index
                for index, bar in enumerate(bars)
                if bar.timestamp == entry_timestamp
            ),
            None
        )

        if entry_index is None:

            warnings.append(
                f"Trade row {row_index + 2}: "
                "no M1 candle exactly matches "
                f"Entry time {entry_time_text}"
            )

            output_rows.append(
                row
            )

            continue

        # ====================================================
        # FIND NEXT SIGNAL
        # ====================================================
        #
        # We use the next row's Signal timestamp.
        #
        # This intentionally includes ALL signals, including
        # signals that may have been ignored by the original
        # backtest.
        # ========================================================

        next_signal_timestamp: int | None = None

        if row_index + 1 < len(trades):

            next_signal_timestamp = (
                _get_signal_timestamp(
                    trades[row_index + 1]
                )
            )

        # ====================================================
        # CALCULATE
        # ====================================================

        (
            mfe,
            tp_hit_index,
            stop_index,
            stop_reason
        ) = _calculate_tp_mfe(
            bars=bars,
            entry_index=entry_index,
            next_signal_timestamp=next_signal_timestamp,
            side=side,
            entry=entry,
            sl=sl,
            tp=tp,
            config=config
        )

        # ====================================================
        # UPDATE MFE ONLY
        # ====================================================

        if mfe is not None:

            row["MFE"] = round(
                mfe,
                6
            )

            row[
                "MFE calculation"
            ] = (
                "TP hit -> maximum favorable "
                "movement -> first entry/SL "
                "visit or next signal"
            )

            if tp_hit_index is not None:

                row[
                    "TP hit time"
                ] = _format_timestamp(
                    bars[
                        tp_hit_index
                    ].timestamp
                )

            if stop_index is not None:

                row[
                    "MFE tracking stopped"
                ] = _format_timestamp(
                    bars[
                        stop_index
                    ].timestamp
                )

            row[
                "MFE stop reason"
            ] = stop_reason

        else:

            row[
                "MFE"
            ] = ""

            row[
                "MFE calculation"
            ] = "TP not found"

            row[
                "TP hit time"
            ] = ""

            row[
                "MFE tracking stopped"
            ] = ""

            row[
                "MFE stop reason"
            ] = "TP_NOT_FOUND"

            warnings.append(
                f"Trade row {row_index + 2}: "
                "TP outcome exists in trade CSV, "
                "but TP could not be found in M1 data "
                "before the next signal."
            )

        output_rows.append(
            row
        )

    # ========================================================
    # OUTPUT PATH
    # ========================================================

    output_path = (
        config.output_path
        if config.output_path is not None
        else trades_path.with_name(
            f"{trades_path.stem}_mfe_recalculated.csv"
        )
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # ========================================================
    # OUTPUT FIELDS
    # ========================================================

    fieldnames: list[str] = []

    for row in output_rows:

        for key in row.keys():

            if key not in fieldnames:

                fieldnames.append(
                    key
                )

    # ========================================================
    # WRITE
    # ========================================================

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as destination:

        writer = csv.DictWriter(
            destination,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        writer.writerows(
            output_rows
        )

    return MFEReport(
        output_path=output_path,
        rows=output_rows,
        warnings=warnings
    )


# ============================================================
# INPUTS
# ============================================================

TRADES_CSV = Path(
    r"D:\Fin\Prod\Live Trading\backtesting\Output\output.csv"
)

FINETIMEFRAME_CSV = Path(
    r"D:\Fin\Prod\Live Trading\public\data\SILVER_M1_IST_20260519_20260826.csv"
)

OUTPUT_CSV = Path(
    r"D:\Fin\Prod\Live Trading\backtesting\Output\output_mfe_recalculated.csv"
)


# ============================================================
# MAIN
# ============================================================

def main() -> None:

    report = recalculate_mfe(

        trades_csv=TRADES_CSV,

        finetimeframe_csv=FINETIMEFRAME_CSV,

        config=MFEConfig(

            sl_distance=0.06,

            tp_distance=0.12,

            include_tp_hit_candle=True,

            output_path=OUTPUT_CSV,
        )
    )

    print()
    print("=" * 70)
    print("TP-BASED MFE RECALCULATION COMPLETE")
    print("=" * 70)

    print(
        f"Output file : "
        f"{report.output_path}"
    )

    tp_count = sum(
        1
        for row in report.rows
        if (
            str(
                row.get(
                    "TP/SL outcome",
                    ""
                )
            ).upper()
            == "TP"
        )
    )

    mfe_count = sum(
        1
        for row in report.rows
        if row.get("MFE", "") != ""
    )

    print(
        f"TP trades   : "
        f"{tp_count}"
    )

    print(
        f"MFE values  : "
        f"{mfe_count}"
    )

    print("=" * 70)

    if report.warnings:

        print()
        print("WARNINGS:")

        for warning in report.warnings:

            print(
                f"WARNING: {warning}"
            )


if __name__ == "__main__":
    main()


TP-BASED MFE RECALCULATION COMPLETE
Output file : D:\Fin\Prod\Live Trading\backtesting\Output\output_mfe_recalculated.csv
TP trades   : 90
MFE values  : 169

WARNINGS:
